### This Note book dedicated for try Build Rag system using Langchain 

### Import Librires

In [ ]:
# Import language models and chains
# from langchain.chains import RetrievalQA
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
# Import document loaders and splitters
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

In [ ]:
# Import embeddings and vector stores
from langchain_community.embeddings import HuggingFaceEmbeddings # Or: from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.vectorstores import VectorStoreRetriever

In [ ]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from tqdm.auto import tqdm
import os
from dotenv import load_dotenv

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
# Import environment variables handlers
import os
from dotenv import load_dotenv

# Load environment variables (e.g., GOOGLE_API_KEY)
load_dotenv()

### Load Dataset from huggingface

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("Amod/mental_health_counseling_conversations")

In [ ]:
ds

### Download data set from hugging face

In [ ]:
from langchain_community.document_loaders.hugging_face_dataset import HuggingFaceDatasetLoader

# Use "Context" as the page content
loader = HuggingFaceDatasetLoader(
    path="Amod/mental_health_counseling_conversations", 
    page_content_column="Context"
)

# Load the documents
documents = loader.load()

# Optional: You can store the "Response" in the document metadata
# if you need to access it later during retrieval
print(documents[0].page_content)

In [ ]:
documents

### Explore Data

In [ ]:
#check repetition and documents
for i, doc in enumerate(documents):
    print(f"Document {i}:")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")
    print("-" * 50)

### The Data contain Dupliactes so we need to make set of data to remove duplicates

In [ ]:
unique_documents = []
seen_combinations = set()

for doc in documents:
    # Create a unique signature for each document using content and response
    content = doc.page_content
    response = doc.metadata.get('Response', '')
    
    unique_signature = (content, response)
    
    # If we haven't seen this exact conversation before, add it to our unique list
    if unique_signature not in seen_combinations:
        seen_combinations.add(unique_signature)
        unique_documents.append(doc)

print(f"Original document count: {len(documents)}")
print(f"Unique document count: {len(unique_documents)}")
print(f"Duplicates removed: {len(documents) - len(unique_documents)}")

# Replace the original list with the unique one for the next steps
documents = unique_documents

### Make Chunnking steps apply `merge` all context in one document and apply `recusrive` chunking 

In [ ]:
# 1. First, merge Context and Response so the RAG knows the answer
merged_documents = []
for doc in documents: # using the unique_documents from your previous step
    patient_text = doc.page_content
    counselor_text = doc.metadata.get('Response', '')
    
    # Combine them into a single string
    combined_text = f"Patient: {patient_text}\n\nCounselor: {counselor_text}"
    
    # Create a new document just passing along any extra metadata without the raw Response duplicate
    new_doc = Document(page_content=combined_text, metadata={"source": "Mental Health Counseling"})
    merged_documents.append(new_doc)

# 2. Setup the Text Splitter
# chunk_size: 1000 characters is roughly 200-250 words (fits well into most Embedding model limits)
# chunk_overlap: 100 characters ensures if a split happens mid-thought, the next chunk has context
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, 
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""]
)

# 3. Split the documents
chunked_documents = text_splitter.split_documents(merged_documents)

print(f"Number of original unique conversations: {len(merged_documents)}")
print(f"Number of chunks after splitting: {len(chunked_documents)}")

In [ ]:
chunked_documents

In [ ]:
# Define where you want the model to be saved
custom_cache_path = "NLP Final Project/cache" # Change this to your desired path

# Initialize embeddings with the cache_folder parameter
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    cache_folder=custom_cache_path,       # <--- SET YOUR CUSTOM PATH HERE
    model_kwargs={'device': 'cpu'}, 
    encode_kwargs={'normalize_embeddings': True} 
)

In [ ]:
# THIS step actually triggers the download of the model to your cache folder
print(f"Loading model into {custom_cache_path} and generating embeddings...")

vectorstore = FAISS.from_documents(chunked_documents, embeddings)

print("Finished! Check your cache folder now.")

# Save the FAISS database to disk so you don't have to embed again later
# vectorstore.save_local("faiss_mental_health_index")

In [ ]:
vectorstore.save_local("faiss_mental_health_index")

In [ ]:
#load saved vectorstore
vectorstore = FAISS.load_local("faiss_mental_health_index", embeddings,allow_dangerous_deserialization=True)

In [ ]:
vectorstore

In [ ]:
from langchain_qdrant import QdrantVectorStore
import os
from dotenv import load_dotenv

load_dotenv() 

print("Generating embeddings and uploading to Qdrant Cloud (with increased timeout)...")

qdrant_url = os.getenv("QDRANT_URL")
qdrant_api_key = os.getenv("QDRANT_API_KEY")

if not qdrant_url:
    raise ValueError("QDRANT_URL is missing!")
vectorstore = QdrantVectorStore.from_documents(
    embedding=embeddings,
    url=qdrant_url,
    api_key=qdrant_api_key,
    collection_name="mental_health_index"
)
from tqdm.auto import tqdm

# Start from index 100 since the first 100 are already uploaded
start_index = 0
batch_size = 100

# Wrap your range() with tqdm()
for i in tqdm(range(start_index, len(chunked_documents), batch_size), desc="Uploading to Qdrant"):
    batch = chunked_documents[i:i + batch_size]
    
    # We remove the print statement so it doesn't break the progress bar visual
    vectorstore.add_documents(documents=batch)
    
print("Finished uploading all remaining batches!")

# vectorstore = QdrantVectorStore.from_documents(
#     documents=chunked_documents, 
#     embedding=embeddings,
#     url=qdrant_url,
#     api_key=qdrant_api_key,
#     collection_name="mental_health_index",
#     # Increase the timeout to 300 seconds (5 minutes)
#     timeout=300, 
#     # Severely limit batch size so Qdrant processes fewer vectors at once
#     batch_size=16 
# )

# print("Finished! Documents are now embedded and stored in Qdrant Cloud.")

In [ ]:
load_dotenv() 

print("Connecting to Qdrant Cloud and uploading batches...")

qdrant_url = os.getenv("QDRANT_URL")
qdrant_api_key = os.getenv("QDRANT_API_KEY")
collection_name = "mental_health_index"

if not qdrant_url:
    raise ValueError("QDRANT_URL is missing!")

# 1. Initialize the Qdrant client directly
client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)

start_index = 0
batch_size = 100

vectorstore = None

# Wrap your range() with tqdm() for the progress bar
for i in tqdm(range(start_index, len(chunked_documents), batch_size), desc="Uploading to Qdrant"):
    batch = chunked_documents[i:i + batch_size]
    
    # If the collection doesn't exist (because it was deleted), use from_documents to create it
    if i == start_index and not client.collection_exists(collection_name):
        vectorstore = QdrantVectorStore.from_documents(
            documents=batch,
            embedding=embeddings,
            url=qdrant_url,
            api_key=qdrant_api_key,
            collection_name=collection_name
        )
    else:
        # If the collection already exists, initialize the vectorstore object and add
        if vectorstore is None:
            vectorstore = QdrantVectorStore(
                client=client,
                collection_name=collection_name,
                embedding=embeddings
            )
        
        # Add the current batch iteratively
        vectorstore.add_documents(documents=batch)
    
print("Finished uploading all batches!")

In [ ]:
qdrant_client = QdrantClient(
    url=os.getenv("QDRANT_URL"), 
    api_key=os.getenv("QDRANT_API_KEY"),
)

print(qdrant_client.get_collections())

In [ ]:
collection_info = qdrant_client.get_collection(collection_name="mental_health_index")
print(f"Total points successfully stored in Qdrant: {collection_info.points_count}")
print(f"Total chunks we tried to upload: {len(chunked_documents)}")

In [ ]:
# Updated imports for LangChain v1
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

### Fiass

In [ ]:
# 1. Prepare the BM25 Retriever
bm25_retriever = BM25Retriever.from_documents(chunked_documents)
bm25_retriever.k = 5

# 2. Prepare the FAISS Vector Retriever
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 3. Combine them using Reciprocal Rank Fusion (RRF)
# The EnsembleRetriever remains functionally identical
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.2, 0.8]
)

# Usage remains the same
query = "I'm feeling really down and can't sleep, what should I do?"
docs = ensemble_retriever.invoke(query)

print(f"\nTop retrieved document:\n{docs[0].page_content}")

### Qudrant

### There is 2 collection from different embadding model we will pick second one which from `BGE m3 from BAAI`

In [ ]:
# 2. Point to Collection A (e.g., your mental health data)
vectorstore_mental_health = QdrantVectorStore(
    client=qdrant_client,
    collection_name="mental_health_index_using_bge-small-en-v1.5",
    embedding=embeddings
)

### Test vector `database` model from qudrant

In [ ]:
# 1. Convert the Qdrant VectorStore into a retriever (Ask for the top 3 most similar results)
qdrant_retriever = vectorstore_mental_health.as_retriever(search_kwargs={"k": 3})

# 2. Define a test query
test_query = "I'm feeling really down and can't sleep, what should I do?"
print(f"Query: {test_query}\n")

# 3. Retrieve the documents!
retrieved_docs = qdrant_retriever.invoke(test_query)

# 4. Print the results to verify
for i, doc in enumerate(retrieved_docs):
    print(f"--- Result {i+1} ---")
    print(doc.page_content)
    print("\n")

### Using Qudrant and hybird retrival

In [ ]:
# 1. Prepare the BM25 Retriever
bm25_retriever = BM25Retriever.from_documents(chunked_documents)
bm25_retriever.k = 5

# 2. Prepare the FAISS Vector Retriever
quadrant = vectorstore_mental_health.as_retriever(search_kwargs={"k": 3})

# 3. Combine them using Reciprocal Rank Fusion (RRF)
# The EnsembleRetriever remains functionally identical
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, quadrant],
    weights=[0.4, 0.6]
)

# Usage remains the same
query = "I'm feeling really down and can't sleep, what should I do?"
docs = ensemble_retriever.invoke(query)

print(f"\nTop retrieved document:\n{docs[0].page_content}")

In [ ]:
# Your existing Qdrant collection as semantic retriever
semantic_retriever = vectorstore_mental_health.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 15,
        "lambda_mult": 0.7
    }
)

# BM25 needs the raw documents — use your chunked_documents
bm25_retriever = BM25Retriever.from_documents(
    chunked_documents,  # ← your existing chunked docs
    k=3
)

# Combine both
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever],
    weights=[0.4, 0.6]
)

# Quick test
docs = hybrid_retriever.invoke("I'm feeling really down and can't sleep, what should I do?")
print(f"Retrieved {len(docs)} docs")
for doc in docs:
    print(doc.page_content[:150])
    print("---")

### Start make RAG chain

In [ ]:
# Import the necessary modern LangChain modules
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# 1. Initialize the LLM
# Make sure your GOOGLE_API_KEY is properly loaded in your environment / .env file
gemini_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", # Use standard gemini-1.5-flash or gemini-pro
    temperature=0.7 # A bit of temperature to sound empathetic and natural
)

In [ ]:
# Note: You need GROQ_API_KEY in your environment for this!
groq_llm = ChatGroq(
    model_name="LLaMA-3.3-70b-versatile", 
    temperature=0
)


In [ ]:
gemini_llm.invoke("what is capital of egypt?")

In [ ]:
groq_llm.invoke("what is capital of egypt?")

In [ ]:
# 2. Create the Prompt Template
# This dictates how the LLM should use the retrieved context to answer the user
system_prompt = (
    "You are a compassionate, professional mental health assistant. "
    "Use the following pieces of retrieved counseling advice to answer the user's question. "
    "If you don't know the answer with the provided context, just say that you don't know, don't try to make up medical advice. "
    "Always maintain an empathetic and supportive tone.\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 3. Create the Document Chain (How to process the retrieved documents)
question_answer_chain = create_stuff_documents_chain(groq_llm, prompt)

# 4. Create the Final Retrieval Chain combining the Retriever and the Document Chain
rag_chain = create_retrieval_chain(ensemble_retriever, question_answer_chain)

print("RAG Chain initialized and ready!")

# --- TEST IT OUT ---
query = "what is length of elphant legs"
response = rag_chain.invoke({"input": query})

print("\n--- User Query ---")
print(query)

# View the retrieved documents used for context
print("\n--- Retrieved Documents (Context) ---")
for i, doc in enumerate(response["context"]):
    print(f"\n[Source {i+1}]")
    print(doc.page_content)

print("\n--- AI Counselor Response ---")
print(response["answer"])

In [ ]:
from warnings import filterwarnings
filterwarnings("ignore")

In [ ]:
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import context_precision, context_recall, answer_relevancy, faithfulness
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
import os


In [ ]:
# 1. Setup the Gemini Judges
# The language model to grade Relevancy and Faithfulness
judge_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [ ]:
# The embedding model to grade Context Precision and Recall
# The embedding model to grade Context Precision and Recall
judge_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001",api_key=os.getenv("GEMINI_API_KEY"))

In [ ]:
judge_embeddings.model

In [ ]:
judge_embeddings.embed_documents(["This is a sample document to embed for evaluation."])

In [ ]:
# 2. Create the Evaluation Dataset
test_questions = [
    "I feel like I have no energy and I just want to lay in bed all day.",
    "My partner hit me the other day, I don't know what to do.",
    "I suddenly feel like I can't breathe, my heart is racing, and I think I'm going to die.",
    "I just lost my mom last week and I can't stop crying. Is there something wrong with me?",
    "Work is so overwhelming right now, I feel like I'm going to completely collapse from stress.",
    "I feel so alone and disconnected, even when I am surrounded by my friends and family.",
    "I constantly feel like I'm not good enough and that everyone is secretly judging me."
]

# Provide concise expected concepts/topics the RAG should cover in its answer
ground_truths = [
    "Depression, sleep hygiene, talk to a doctor or therapist.", 
    "Domestic violence hotline, physical safety planning, seeking immediate shelter.",
    "Panic attack symptoms, grounding techniques, deep breathing exercises.",
    "Grief and loss, validating the mourning process, it is normal to cry.",
    "Burnout, setting boundaries, stress management, taking time off.",
    "Social isolation, seeking meaningful connections, expressing vulnerability.",
    "Imposter syndrome, low self-esteem, challenging negative thoughts, cognitive behavioral therapy."
]

In [ ]:
answers = []
contexts = []

In [ ]:
for q in test_questions:
    retrieved_docs = ensemble_retriever.invoke(q)
    contexts.append([doc.page_content for doc in retrieved_docs])
    response = rag_chain.invoke({"input": q})
    answers.append(response["answer"])

# Format for Ragas
dataset = Dataset.from_dict({
    "question": test_questions,
    "answer": answers,
    "contexts": contexts,
    "reference": ground_truths
})

In [ ]:
dataset

In [ ]:
# 3. RUN EVALUATION USING GEMINI
print("Running Ragas Evaluation with Gemini...")
score = evaluate(
    dataset,
    metrics=[
        context_precision,
        context_recall,
        answer_relevancy,
        faithfulness
    ],
    llm=judge_llm,               # Tell Ragas to use Gemini
    embeddings=judge_embeddings  # Tell Ragas to use Google Embeddings
)

print("\nDetailed DataFrame:")
display(score.to_pandas())

In [ ]:
# 3. RUN EVALUATION USING GEMINI
print("Running Ragas Evaluation with Gemini...")
score = evaluate(
    dataset,
    metrics=[
        context_precision,
        context_recall,
        answer_relevancy,
        faithfulness
    ],
    llm=groq_llm,               # Tell Ragas to use Gemini
    embeddings=judge_embeddings  # Tell Ragas to use Google Embeddings
)

print("\nDetailed DataFrame:")
display(score.to_pandas())

In [ ]:
# 3. RUN EVALUATION USING GEMINI
print("Running Ragas Evaluation with Gemini...")
score = evaluate(
    dataset,
    metrics=[
        context_precision,
        context_recall,
        answer_relevancy,
        faithfulness
    ],
    llm=groq_llm,               # Tell Ragas to use Gemini
    embeddings=judge_embeddings  # Tell Ragas to use Google Embeddings
)

print("\nDetailed DataFrame:")
display(score.to_pandas())